# Customer 360 Intelligence - Practical Machine Learning Workshop
### NileConnect | Part G - Customer 360 Integration

## 1. Imports

In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor, IsolationForest
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 2. Load Dataset

In [2]:
df = pd.read_csv("customer_360_ml_workshop.csv")
df.head()

,CustomerID,Age,Region,TenureMonths,ContractType,InternetType,MonthlyUsageGB,MonthlyChargeEGP,NumServices,SupportCalls6M,LatePayments12M,PaperlessBilling,AutoPay,SatisfactionScore,Future12MRevenueEGP,Churn
0,CUST-00001,22,Alexandria,23,Month-to-month,DSL,317.9,580.48,5,5,0,Yes,Yes,5.0,6287.08,Yes
1,CUST-00002,59,Upper Egypt,9,Month-to-month,4G/5G,359.3,434.36,1,1,1,No,Yes,4.0,4619.91,No
2,CUST-00003,52,Giza,21,Month-to-month,DSL,176.9,391.35,3,5,1,Yes,Yes,5.0,4512.25,No
3,CUST-00004,41,Giza,12,Month-to-month,DSL,312.5,420.40,3,2,1,No,Yes,5.0,3809.65,No
4,CUST-00005,40,Delta,21,Month-to-month,4G/5G,191.6,462.73,3,4,0,No,Yes,3.0,4731.09,No


## 3. Churn Probability (from Part C)

The tuned Random Forest from Part C (`n_estimators=300`, `max_depth=5`, `min_samples_split=2`, `class_weight="balanced"`, found with `GridSearchCV`) is refit on the full dataset and used to score every customer's churn probability.

**Note:** because the model is trained with `class_weight="balanced"` to counter the 18% churn rate, these probabilities are better read as a **relative risk ranking** between customers than as a literal "X% chance to churn" - they are shifted upward on average compared to the true churn rate, by design, so that the model does not simply ignore the minority class.

In [3]:
X_churn = df.drop(columns=["CustomerID", "Future12MRevenueEGP", "Churn"])
y_churn = (df["Churn"] == "Yes").astype(int)

numeric_features = X_churn.select_dtypes(include="number").columns.tolist()
categorical_features = X_churn.select_dtypes(exclude="number").columns.tolist()

churn_preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_features)
])

churn_model = RandomForestClassifier(
    n_estimators=300, max_depth=5, min_samples_split=2, class_weight="balanced", random_state=42
)
churn_pipeline = Pipeline(steps=[("preprocessor", churn_preprocessor), ("model", churn_model)])
churn_pipeline.fit(X_churn, y_churn)

df["ChurnProbability"] = churn_pipeline.predict_proba(X_churn)[:, 1]
df["ChurnProbability"].describe()

count    3000.000000
mean        0.431449
std         0.168681
min         0.129685
25%         0.269238
50%         0.442225
75%         0.580574
max         0.769944
Name: ChurnProbability, dtype: float64

## 4. Predicted 12-Month Revenue (from Part D)

The Gradient Boosting Regressor from Part D - the best-performing model in the Part D comparison - is refit on the full dataset and used to predict `Future12MRevenueEGP` for every customer.

In [4]:
X_revenue = df.drop(columns=["CustomerID", "Future12MRevenueEGP", "Churn"])
y_revenue = df["Future12MRevenueEGP"]

revenue_preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), categorical_features)
])

revenue_model = GradientBoostingRegressor(random_state=42)
revenue_pipeline = Pipeline(steps=[("preprocessor", revenue_preprocessor), ("model", revenue_model)])
revenue_pipeline.fit(X_revenue, y_revenue)

df["Predicted12MRevenueEGP"] = revenue_pipeline.predict(X_revenue)
df["Predicted12MRevenueEGP"].describe()

count     3000.000000
mean      5361.156050
std       1399.982483
min       1851.681085
25%       4367.210022
50%       5310.797479
75%       6240.297296
max      10821.627748
Name: Predicted12MRevenueEGP, dtype: float64

## 5. Customer Segment (from Part E)

The same K-Means setup from Part E (`k=4`, `random_state=42`) on the same standardized behavioral/value variables reproduces the same four segments and business names used throughout this workshop.

In [5]:
cluster_features = ["TenureMonths", "MonthlyUsageGB", "MonthlyChargeEGP", "NumServices",
                    "SupportCalls6M", "LatePayments12M", "SatisfactionScore"]

segmentation_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
X_scaled = segmentation_preprocessor.fit_transform(df[cluster_features])

segmentation_model = KMeans(n_clusters=4, random_state=42, n_init=10)
df["Cluster_KMeans"] = segmentation_model.fit_predict(X_scaled)

cluster_business_names = {
    0: "At-Risk / Late-Payment Customers",
    1: "Loyal Long-Tenured Customers",
    2: "Low-Value Basic Users",
    3: "High-Value Power Users"
}
df["CustomerSegment"] = df["Cluster_KMeans"].map(cluster_business_names)

df["CustomerSegment"].value_counts()

CustomerSegment
Low-Value Basic Users               1041
High-Value Power Users               979
At-Risk / Late-Payment Customers     518
Loyal Long-Tenured Customers         462
Name: count, dtype: int64

## 6. Anomaly Flag (from Part F)

The same Isolation Forest setup from Part F (`contamination=0.05`) flags the 5% of customers whose behavioral pattern is most unusual, exactly as in Part F.

In [6]:
anomaly_model = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
anomaly_labels = anomaly_model.fit_predict(X_scaled)

df["AnomalyFlag"] = np.where(anomaly_labels == -1, "Yes", "No")
df["AnomalyFlag"].value_counts()

AnomalyFlag
No     2850
Yes     150
Name: count, dtype: int64

## 7. Retention-Priority Flag

### The rule

**A customer is flagged `RetentionPriorityFlag = "Yes"` if both of the following are true:**

1. `ChurnProbability >= 0.5` - the model considers the customer more likely than not to churn.
2. `Predicted12MRevenueEGP` is at or above the **median** predicted revenue across all customers - the customer is at least average value, not a low-value customer.

### Why this rule is reasonable

- Flagging on churn risk alone would include customers the business would not necessarily want to spend retention budget on (for example, a low-value customer unlikely to be profitable even if retained).
- Flagging on revenue alone would include loyal, low-risk customers who do not need a retention intervention at all.
- **Requiring both** targets the customers where a retention action has both a clear reason (real churn risk) and a clear payoff (meaningful revenue at stake) - the customers a retention budget should reach first.
- The threshold of 0.5 on `ChurnProbability` is the natural midpoint of a probability; the median of `Predicted12MRevenueEGP` is a simple, data-driven value cutoff that does not need to be guessed.

In [7]:
median_predicted_revenue = df["Predicted12MRevenueEGP"].median()
print("Median predicted 12-month revenue:", round(median_predicted_revenue, 2))

df["RetentionPriorityFlag"] = np.where(
    (df["ChurnProbability"] >= 0.5) & (df["Predicted12MRevenueEGP"] >= median_predicted_revenue),
    "Yes", "No"
)

df["RetentionPriorityFlag"].value_counts()

Median predicted 12-month revenue: 5310.8


RetentionPriorityFlag
No     2520
Yes     480
Name: count, dtype: int64

### Checking the rule's behavior

Before trusting the rule, its output is checked against the segments and the anomaly flag it was not built from.

In [8]:
print("Average ChurnProbability and Predicted12MRevenueEGP by retention flag:")
print(df.groupby("RetentionPriorityFlag")[["ChurnProbability", "Predicted12MRevenueEGP"]].mean().round(2))
print()
print("Retention-priority customers by segment:")
print(pd.crosstab(df["CustomerSegment"], df["RetentionPriorityFlag"]))
print()
print("Retention-priority customers by anomaly flag:")
print(pd.crosstab(df["AnomalyFlag"], df["RetentionPriorityFlag"]))

Average ChurnProbability and Predicted12MRevenueEGP by retention flag:
                       ChurnProbability  Predicted12MRevenueEGP
RetentionPriorityFlag                                          
No                                 0.40                 5239.88
Yes                                0.61                 5997.85

Retention-priority customers by segment:
RetentionPriorityFlag               No  Yes
CustomerSegment                            
At-Risk / Late-Payment Customers   424   94
High-Value Power Users             622  357
Low-Value Basic Users             1039    2
Loyal Long-Tenured Customers       435   27

Retention-priority customers by anomaly flag:
RetentionPriorityFlag    No  Yes
AnomalyFlag                     
No                     2391  459
Yes                     129   21


## 8. Final Customer 360 Table

One row per customer, with every output this workshop produced.

In [9]:
final_columns = ["CustomerID", "ChurnProbability", "Predicted12MRevenueEGP",
                 "CustomerSegment", "AnomalyFlag", "RetentionPriorityFlag"]

customer_360_table = df[final_columns].copy()
customer_360_table["ChurnProbability"] = customer_360_table["ChurnProbability"].round(4)
customer_360_table["Predicted12MRevenueEGP"] = customer_360_table["Predicted12MRevenueEGP"].round(2)

print("Final table shape:", customer_360_table.shape)
customer_360_table.head(10)

Final table shape: (3000, 6)


,CustomerID,ChurnProbability,Predicted12MRevenueEGP,CustomerSegment,AnomalyFlag,RetentionPriorityFlag
0,CUST-00001,0.5728,5943.81,High-Value Power Users,No,Yes
1,CUST-00002,0.5124,4222.67,Low-Value Basic Users,No,No
2,CUST-00003,0.5225,4080.92,Low-Value Basic Users,No,No
3,CUST-00004,0.4960,4205.31,Low-Value Basic Users,No,No
4,CUST-00005,0.5563,4686.33,Low-Value Basic Users,No,No
5,CUST-00006,0.2596,5453.65,Low-Value Basic Users,No,No
6,CUST-00007,0.6507,3941.03,At-Risk / Late-Payment Customers,No,No
7,CUST-00008,0.2318,5692.48,Loyal Long-Tenured Customers,No,No
8,CUST-00009,0.2172,3567.45,Low-Value Basic Users,No,No
9,CUST-00010,0.3328,4236.94,Low-Value Basic Users,No,No


## 9. Saving the Output

In [10]:
import os

os.makedirs("outputs", exist_ok=True)
customer_360_table.to_csv("outputs/customer_360_predictions.csv", index=False)

print("Saved to outputs/customer_360_predictions.csv")
print("Rows:", len(customer_360_table), "| Columns:", list(customer_360_table.columns))

Saved to outputs/customer_360_predictions.csv
Rows: 3000 | Columns: ['CustomerID', 'ChurnProbability', 'Predicted12MRevenueEGP', 'CustomerSegment', 'AnomalyFlag', 'RetentionPriorityFlag']


## 10. Final Interpretation

### What the integration shows

- **480 of 3,000 customers (16%)** are flagged `RetentionPriorityFlag = "Yes"`. Their average churn probability is 0.61 (vs 0.40 for the rest) and their average predicted revenue is 5,997.85 EGP (vs 5,239.88 EGP for the rest) - confirming the rule is doing what it was designed to do: finding customers who are both risky and valuable.
- **The retention-priority list is not just the At-Risk segment relabeled.** Of the 480 flagged customers, 357 (74%) come from the **High-Value Power Users** segment and only 94 from the **At-Risk / Late-Payment Customers** segment. This is a genuinely useful finding: the rule surfaces high-value customers who the churn model considers at real risk - exactly the customers a segment label alone would not have flagged, since "High-Value Power Users" as a group has a below-average churn rate.
- Almost no **Low-Value Basic Users** are flagged (2 of 1,041), because they rarely clear the revenue bar - which is the intended behavior of requiring both conditions.
- The anomaly flag and the retention-priority flag largely disagree (only 21 customers are both), which makes sense: they are built to catch different things - Isolation Forest looks for unusual behavioral *patterns*, while the retention rule looks for a combination of churn risk and revenue. A customer can be perfectly "normal" in behavior and still be a retention priority, or be behaviorally unusual without being a near-term churn risk.

### Business meaning

- This table turns four separate models into one prioritized action list: **start with the 480 `RetentionPriorityFlag = "Yes"` customers**, especially the High-Value Power Users among them, since they combine the clearest business case (high revenue) with a real, model-estimated risk of leaving.
- The 150 customers with `AnomalyFlag = "Yes"` are a separate, smaller list for manual account review (see Part F) - they are not automatically retention priorities, and not automatically fraud.
- `CustomerSegment` gives context for *how* to retain a flagged customer (for example, a longer-contract offer for a retention-priority customer who is also in the At-Risk / Late-Payment segment), even though the flag itself does not depend on the segment.
- As with every part of this workshop, these are model outputs on a single snapshot of synthetic data; a production deployment would refresh these scores regularly and track how the flagged customers actually behave over time.

## 11. Final Checklist

**Part G - Customer 360 Integration**
- [x] `CustomerID` included
- [x] Churn probability included (Part C model, refit on full data)
- [x] Predicted 12-month revenue included (Part D model, refit on full data)
- [x] Customer segment included (Part E K-Means segments, with business names)
- [x] Anomaly flag included (Part F Isolation Forest)
- [x] Retention-priority flag included, with a reasonable rule defined and explained
- [x] Final output saved to `outputs/customer_360_predictions.csv`